In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import time

from google.colab import drive
drive.mount('/content/drive')

from torchvision import datasets
data_path = '/content/drive/MyDrive/Machine Learning/data/'

cifar10 = datasets.CIFAR10(data_path, train=True, download=True)
cifar10_val = datasets.CIFAR10(data_path, train=False, download=True)

from torchvision import transforms
to_tensor = transforms.ToTensor()

tensor_cifar10 = datasets.CIFAR10(data_path, train=True, download=False,
                                  transform=to_tensor)

imgs = torch.stack([img_t for img_t, _ in tensor_cifar10], dim=3)
imgs.shape
print(imgs.view(3,-1).mean(dim=1))
print(imgs.view(3,-1).std(dim=1))

transformed_cifar10 = datasets.CIFAR10(
    data_path, train=True, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616))
]))

transformed_cifar10_val = datasets.CIFAR10(
    data_path, train=False, download=False,
    transform=transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465),
                             (0.2470, 0.2435, 0.2616))
]))

train_loader = torch.utils.data.DataLoader(transformed_cifar10, batch_size=64,
                                           shuffle=True)
val_loader = torch.utils.data.DataLoader(transformed_cifar10_val, batch_size=64,
                                          shuffle=False)

model = nn.Sequential(
    nn.Linear(3072, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10),
    nn.LogSoftmax(dim=1)
)

# model = nn.Sequential(
#     nn.Linear(3072, 512),
#     nn.Tanh(),
#     nn.Linear(512, 512),
#     nn.Tanh(),
#     nn.Linear(512, 10),
#     nn.LogSoftmax(dim=1)
# )

# model = nn.Sequential(
#     nn.Linear(3072, 1024),
#     nn.ReLU(),
#     nn.Linear(1024, 1024),
#     nn.ReLU(),
#     nn.Linear(1024, 10),
#     nn.LogSoftmax(dim=1)
# )

# model = nn.Sequential(
#     nn.Linear(3072, 512),
#     nn.Sigmoid(),
#     nn.Linear(512, 512),
#     nn.Sigmoid(),
#     nn.Linear(512, 10),
#     nn.LogSoftmax(dim=1)
# )

# model = nn.Sequential(
#     nn.Linear(3072, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 10),
#     nn.LogSoftmax(dim=1)
# )

# model = nn.Sequential(
#     nn.Linear(3072, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 512),
#     nn.ReLU(),
#     nn.Linear(512, 10),
#     nn.LogSoftmax(dim=1)
# )

optimizer = optim.SGD(model.parameters(), lr=1e-2)

loss_fn = nn.NLLLoss()

n_epochs = 80

numel_list = [p.numel() for p in model.parameters() if p.requires_grad]
print("Trainable parameters:", sum(numel_list))

val_accs = []
train_accs = []
train_losses = []

start = time.time()

for epoch in range(n_epochs):
    total_loss = 0
    correct = 0
    total = 0
    for img, label in train_loader:
        batch_size = img.shape[0]
        out = model(img.view(batch_size, -1))
        loss = loss_fn(out, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        _, predicted = torch.max(out, dim=1)
        total += label.shape[0]
        correct += int((predicted == label).sum())

    train_losses.append(total_loss / len(train_loader))
    train_accs.append(correct / total)
    correct = 0
    total = 0

    with torch.no_grad():
      for img, label in val_loader:
        batch_size = img.shape[0]
        out = model(img.view(batch_size, -1))
        _, predicted = torch.max(out, dim=1)
        total += label.shape[0]
        correct += int((predicted == label).sum())
    val_accs.append(correct / total)

    print("Epoch: %d, Loss: %f" % (epoch, total_loss / len(train_loader)))

print("Training time: %.1f seconds" % (time.time() - start))
print("Training Accuracy: %f" % train_accs[-1])
print("Validation Accuracy: %f" % val_accs[-1])
print("Training Loss: %f" % train_losses[-1])

plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss")
plt.show()

plt.plot(train_accs, label="Training")
plt.plot(val_accs, label="Evaluation")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Training vs Evaluation Accuracy")
plt.legend()
plt.show()

Mounted at /content/drive


KeyboardInterrupt: 